In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from torch.distributions import NegativeBinomial
import itertools

This code was ran individually for each dataset due to memory (RAM) constraints
Please change file names in accordance to the imported dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU available: {torch.cuda.is_available()}")

In [ ]:
class MIWAE_NB(nn.Module):
    def __init__(self, input_size, latent_size=32, hidden_size=512, K=100):
        super(MIWAE_NB, self).__init__()
        self.input_size = input_size
        self.latent_size = latent_size
        self.hidden_size = hidden_size
        self.K = K

        """
        Encoder
        Takes observed data and compresses it into a small latent space (z) with fewer dimensions
        Input is library-size normalised log counts and mask
        Returns 2 vectors, mean (mu) and log-variance (logvar)
        mu and logvar define a probability distribution over z
        z is calculated during reparameterisation

        5%: hidden_size = 256
        10% and 30%: hidden_size = 512
        """
        self.encoder = nn.Sequential(
            nn.Linear(input_size*2, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size // 2), 
            nn.GELU()
        )
        self.enc_mu = nn.Linear(hidden_size // 2, latent_size) # where the encoder thinks z should be centred
        self.enc_logvar = nn.Linear(hidden_size // 2, latent_size) # how uncertain the encoder is about that location

        """
        Decoder
        Maps K latent vectors of z back to the gene expression space
        Input is z calculated during reparameterisation
        For each z two parameters of the NB distribution are produced
        Returns log NB mean (mu) and log dispersion (r) for each gene
        Predicting mu and r and log space ensures these values are positive for NB distribution
        """
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, hidden_size),
            nn.LayerNorm(hidden_size), 
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU()
        )
        self.dec_log_mu = nn.Linear(hidden_size, input_size) # log NB mean
        self.dec_log_r = nn.Linear(hidden_size, input_size) # log NB dispersion

    def encode(self, x, mask): 
        """
        h (hidden vector): (n_samples, hidden size)
        mu: (n_samples, latent_size)
        logvar: (n_samples, latent_size)
        """
        h = self.encoder(torch.cat([x, mask], dim=-1)) # concatenates data with mask dim(n_samples, n_genes*2)
        mu = self.enc_mu(h)
        logvar = torch.clamp(self.enc_logvar(h), min=-4, max=4)
        return mu, logvar
    
    def decode(self, z):
        """
        z: (n_samples*K, latent_size)
        h (hidden vector): (n_samples*K, hidden_size)
        log_mu/mu: (n_samples*K, n_genes)
        log_r/mu: (n_samples*K, n_genes)
        Clamping prevents overflow and underflow, crashing NB likelihood 
        and keeping predictions in a realistic range
        """
        h = self.decoder(z)
        log_mu = torch.clamp(self.dec_log_mu(h), min=-5, max=15)
        log_r = torch.clamp(self.dec_log_r(h),  min=-5, max=15)
        mu = torch.exp(log_mu) # NB mean 
        r = torch.exp(log_r) # NB dispersion
        return mu, r
    
    def reparameterise(self, mu, logvar):
        """
        mu:  (n_samples, K, latent_size)
        std: (n_samples, K, latent_size)
        z:   (n_samples, K, latent_size)
        """
        std = torch.exp(0.5 * logvar)
        z = mu + std * torch.randn_like(std)
        return std, z
    
    def nb_log_prob(self, x, mu, r):
        """
        Negative Binomial log probability
        Computes how likely the observed x are given in the decoders predicted mu and r
        Higher log prob, the more likely the predictions are
        x: raw integer counts
        mu: predicted mean per gene
        r: predicted dispersion per gene
        logits: signal to noise ratio
            - lower: tighter distribution, low overdispersion, Poisson-like
            - higher: wider distribution, higher dispersion
            - log prevents extreme values that could cause numerical instability in the log prob
            - logits = log(mu) - log(r) = log(mu/r)
        Returns log probability per gene per sample per K: (n_samples, K, n_genes)/(B, K, D)
        """
        mu = torch.clamp(mu, min=1e-6, max=1e6) # must be positive and non-zero
        r = torch.clamp(r,  min=1e-6, max=1e6)
        x = torch.clamp(x,  min=0) # x cannot be negative 
        logits = torch.clamp(torch.log(mu) - torch.log(r), min=-10, max=10)
        return NegativeBinomial(total_count=r, logits=logits).log_prob(x)
    
    def calculate_weights(self, x, mask, z, enc_mu, dec_mu, r, logvar, std):
        """"
        Predicts NB likelihood per gene per sample per K, for observed values only
        Calculates importance weight per latent sample (z): how well each latent sample explains x

        logp(x|z): how well the decoder reconstructs the observed values 
                   (missing values are masked (0) so they don't influence training - MCAR)
        logp(z): standard normal prior, how likely z is under the prior (Normal (N(0,1)))
                 acts as a regulariser, keeping latent space structured and continuous
        logq(z|x): posterior, how likely z is under the encoder distribution (Gaussian (N(mu,std^2)))

        Importance weights: w = p(x|z) * p(z) / q(z|x)
        Log importance weight: log_w = log_px_z + log_pz - log_qz_x
        """
        # logp(x_obs|z)
        log_px_z = self.nb_log_prob(x, dec_mu, r)
        log_px_z = (log_px_z * mask).sum(-1)

        # logp(z)
        log_pz = -0.5 * (z ** 2 + np.log(2 * np.pi)).sum(-1)

        # logq(z|x)
        log_qz_x = -0.5 * (((z - enc_mu) / std) ** 2 + logvar + np.log(2 * np.pi)).sum(-1)

        # Calculate log importance weights
        log_w = log_px_z + log_pz - log_qz_x
        return log_w        

    def forward(self, x_input, x_raw, mask):
        """
        1. Encode the observed data (x) to get the params of the posterior (q(z|x))
        2. Draw K latent samples per datapoint (z) using reparameterisation trick
        3. Decode each latent sample to predict the NB params (mean and dispersion) per gene
        4. Compute log importance weights for each latent sample (log_w)
        5. Compute importance-weighted ELBO
        6. Return negative ELBO as the training loss
        """
        B, D = x_input.shape # B: batch size (number of samples), D: dimensions (number of genes)
        K = self.K 

        # Encode, mean and log variance per latent space (z)
        enc_mu, logvar = self.encode(x_input, mask)
        enc_mu = enc_mu.unsqueeze(1).expand(B, K, -1) # (B, latent_size) -> (B, K, latent_size)
        logvar = logvar.unsqueeze(1).expand(B, K, -1)

        # Sample K latent vectors via reparameterisation
        std, z = self.reparameterise(enc_mu, logvar)

        # Decode, predict NB mean and dispersion per gene
        z_flat = z.reshape(B*K, -1) # flattens for decoding
        dec_mu, r = self.decode(z_flat)
        dec_mu = dec_mu.reshape(B, K, D) # restores original dimensions
        r = r.reshape(B, K, D)

        # Expand raw counts and mask for K samples
        x_raw_exp = x_raw.unsqueeze(1).expand(B, K, D)
        mask_exp = mask.unsqueeze(1).expand(B, K, D)

        # Calculate log weights
        log_w = self.calculate_weights(x_raw_exp, mask_exp, z, enc_mu, dec_mu, r, logvar, std)

        # Calculate loss: takes the mean of each weight
        # negative as optimisers minimise loss (maximises ELBO)
        # logsumexp(log_w, dim=1) = log of the sum of importance weights
        # -log(K) = log of the average importance weight
        loss = -torch.mean(torch.logsumexp(log_w, dim=1) - np.log(K))

        # Penalise predicted near-zero means where observed values are non-zero
        nonzero_mask = (x_raw > 0).float().unsqueeze(1).expand(B, K, D)
        pred_near_zero = (dec_mu < 1.0).float()
        zero_penalty = (nonzero_mask * pred_near_zero).mean()
        loss = loss + 0.1 * zero_penalty 

        return loss
    
    @torch.no_grad()
    def impute(self, x_input, x_raw, mask, L=100):
        """
        Impute missing values
        L: latent variables per datapoint, weighted to impute missing data point

        1. Encode the observed data (x) to get the params of the posterior (q(z|x))
        2. Draw L latent samples (z_l) 
        3. Decode each z_l to predict the full data vector, including missing datapoints
        4. Compute importance weights 
        5. Impute values as the weighted average of the decoded predictions
        6. Replace only the missing datapoints with imputed values
        """
        self.eval()
        B, D = x_input.shape
        enc_mu, logvar = self.encode(x_input, mask)
        enc_mu = enc_mu.unsqueeze(1).expand(B, L, -1)
        logvar = logvar.unsqueeze(1).expand(B, L, -1)
        std, z = self.reparameterise(enc_mu, logvar)

        dec_mu, r = self.decode(z.reshape(B * L, -1))
        dec_mu = dec_mu.reshape(B, L, D)
        r = r.reshape(B, L, D)

        x_raw_exp = x_raw.unsqueeze(1).expand(B, L, D)
        mask_exp = mask.unsqueeze(1).expand(B, L, D)

        log_w = self.calculate_weights(x_raw_exp, mask_exp, z, enc_mu, dec_mu, r, logvar, std)
        w = torch.softmax(log_w, dim=1)
        imputed = (w.unsqueeze(-1) * dec_mu).sum(1)


        # Keep observed values, fill missing with imputed
        x_raw_np = x_raw.cpu().numpy()
        mask_np = mask.cpu().numpy()
        return x_raw_np * mask_np + imputed.cpu().numpy() * (1 - mask_np)

In [ ]:
def train_miwae_nb(X_input, X_raw, mask,
                   epochs = 3000,
                   lr = 0.0005,
                   batch_size = 16,
                   latent_size = 16,
                   hidden_size = 512,
                   K = 100,
                   patience = 50,
                   log_every = 50
                   ):
    """ 
    Train MIWAE NB model:

    X_input: log1p library0size normalised counts (n_samples, n_genes)
    X_raw: raw integer counts (n_samples, n_genes)
    mask: observation mask (n_samples, n_genes), 1 = observed, 0 = missing
    epochs: max number of epochs
    lr: initial learning rate
    batch_size: number of samples per batch
    K: number of importance samples per datapoint
    patience: early stopping, number of epochs without improvement before stoppining training
    log_every: print progress every this many epochs

    1. Initialise MIWAE model
    2. Adam Optimiser: adaptive learning rate, 
        weight_decay: penalises large weights slightly to prevent overfitting at small n
    3.  Scheduler: monitors loss after each epoch
        - ReduceLROnPlateau: reduces learning rate when loss stops improving
        - factor = 0.5, halves the learning rate when triggered
        - patience = 50, waits 50 epochs of no improvement before reducing
    4. Loader: handles batching
    5. Training loop: train model
        - Early stopping: when no_improve = patience, training stops (no improvement after 50 epochs),
    """
    N,D = X_input.shape

    # Initialise MIWAE
    model = MIWAE_NB(input_size=D, latent_size=latent_size, hidden_size=hidden_size, K=K
                     ).to(device) #run on gpu

    # Optimiser
    optimiser = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    # Learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, 
                                                           patience=50, verbose=True)
    
    # Convert numpy arrays to tensor 
    X_input_t = torch.tensor(X_input, dtype=torch.float32).to(device)
    X_raw_t = torch.tensor(X_raw, dtype=torch.float32).to(device)
    mask_t = torch.tensor(mask, dtype=torch.float32).to(device)

    # Loader
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_input_t, X_raw_t, mask_t),
        batch_size=batch_size, shuffle=True
    )
    
    best_loss = float('inf')
    best_state = None
    no_improve = 0

    # Training loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        valid_batch = 0

        for x_in, x_raw, mb in loader:
            optimiser.zero_grad() # clears gradients from previous batch
            loss = model(x_in, x_raw, mb) # calls model.forward(), returns loss

            # Skip batch if loss collapsed or exploded (nan or inf)
            if torch.isnan(loss) or torch.isinf(loss):
                continue

            # Computed gradients of loss via backpropagation, used by optimiser.step() to update weights
            loss.backward()

            # Clips gradients if their norm exceeds 0.5, prevents exploding gradients
            # NB likelihood can produce large gradients which can destabilise training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimiser.step() # updates all model parameters using computed gradients
            total_loss += loss.item() # .item() converts tensor to float
            valid_batch += 1

        # If every batch had nan/inf loss - stop training
        if valid_batch == 0:
            print(f"Epoch {epoch+1}: all batches invalid, stopping")
            break
    
        # Average loss across all valid batches 
        epoch_loss = total_loss/valid_batch
        scheduler.step(epoch_loss) # scheduler tracks whether loss is improving (reduces lr if no improvement)

        # Early stopping
        if epoch_loss < best_loss:
            best_loss = epoch_loss # update best_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()} # keep track of all best model weights
            no_improve = 0 # reset on improvement
        else:
            no_improve += 1 # loss did not improve  
        
        # Prints progress every log_every epochs
        if (epoch + 1) % log_every == 0:
            print(f"Epoch {epoch+1:4d}/{epochs}"
                  f"  Loss: {epoch_loss:.2f}"
                  f"  Best: {best_loss:.2f}"
                  f"  LR: {optimiser.param_groups[0]['lr']:.2e}" # optimiser reads current lr
                  f"  No improve: {no_improve}/{patience}")
        
        # Stop training if no improvement after 50 epochs
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Restores to the best model after training ends (not the last which may have overfit)
    if best_state:
        model.load_state_dict(best_state)
        print(f"Restored best model: {best_loss:.2f}")

    return model

In [ ]:
def create_validation_mask(mask, holdout_fraction=0.1, seed=42):
    """
    Creates a training mask by holding out a fraction of observed values
    holdout_fraction: proportion of observed values held out for validation
    Returns train_mask: 1 = observed for training, 0 = missing, and holdout_idx
    """
    np.random.seed(seed) # seed for reproducability 
    observed_idx = np.argwhere(mask == 1)
    n_holdout = int(len(observed_idx) * holdout_fraction)
    holdout_idx = observed_idx[
        np.random.choice(len(observed_idx), n_holdout, replace=False)
    ]
    train_mask = mask.copy()
    for i, j in holdout_idx:
        train_mask[i, j] = 0
    return train_mask, holdout_idx

In [ ]:
def evaluate_imputation(X_true, X_imputed, holdout_idx):
    """
    Evaluation metrics for all imputed values

    MAE: average of how far of the estimates are
    RMSE: average error, more sensitive to large errors
    NRSME (by std): normalised RMSE by standard devisation (error relative to the varability) 
    Spearman Correlation: correlation between true values and imputed values
    """
    true_vals = X_true[holdout_idx[:, 0], holdout_idx[:, 1]]
    imputed_vals = X_imputed[holdout_idx[:, 0], holdout_idx[:, 1]]
    mae = np.mean(np.abs(true_vals - imputed_vals))
    rmse = np.sqrt(np.mean((true_vals - imputed_vals) ** 2))
    nrmse_std = rmse / (np.std(true_vals) + 1e-8)
    spearman_corr = spearmanr(true_vals, imputed_vals)

    print("\nEvaluation metrics")
    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"NRMSE (by std): {nrmse_std:.4f}")
    print(f"Spearman r: {spearman_corr.correlation:.4f}")
    print(f"P-Value: {spearman_corr.pvalue:.4f}")
    
    return mae, rmse, nrmse_std, spearman_corr.correlation

def evaluate_imputation_nonzero(X_true, X_imputed, holdout_idx):
    """
    Evaluation metrics for non-zero values
    """
    true_vals = X_true[holdout_idx[:, 0], holdout_idx[:, 1]]
    imputed_vals = X_imputed[holdout_idx[:, 0], holdout_idx[:, 1]]
    nonzero_mask = true_vals > 0
    true_nz = true_vals[nonzero_mask]
    imputed_nz = imputed_vals[nonzero_mask]
    print("\nEvaluation metrics for non-zero values")
    print(f"Non-zero held out: {nonzero_mask.sum()} ({nonzero_mask.mean()*100:.1f}%)")
    print(f"Zero held out: {(~nonzero_mask).sum()} ({(~nonzero_mask).mean()*100:.1f}%)")
    
    mae = np.mean(np.abs(true_nz - imputed_nz))
    rmse = np.sqrt(np.mean((true_nz - imputed_nz) ** 2))
    nrmse_std = rmse / (np.std(true_nz) + 1e-8)
    spearman_corr = spearmanr(true_nz, imputed_nz)

    print(f"MAE: {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"NRMSE (by std): {nrmse_std:.4f}")
    print(f"Spearman r: {spearman_corr.correlation:.4f}")
    print(f"P-Value: {spearman_corr.pvalue:.4f}")
    
    return mae, rmse, nrmse_std, spearman_corr.correlation
    
def evaluate_imputation_logspace(X_true, X_imputed, holdout_idx):
    """
    Evaluation in log space

    More biologically meaningful for RNAseq data
    """
    true_vals = X_true[holdout_idx[:, 0], holdout_idx[:, 1]]
    imputed_vals = X_imputed[holdout_idx[:, 0], holdout_idx[:, 1]]
    true_log = np.log1p(true_vals.astype(float))
    imputed_log = np.log1p(imputed_vals.astype(float))
    mae = np.mean(np.abs(true_log - imputed_log))
    rmse = np.sqrt(np.mean((true_log - imputed_log) ** 2))
    corr = spearmanr(true_log, imputed_log)
    print(f"\nLog Space Evaluation")
    print(f"MAE (log): {mae:.4f}")
    print(f"RMSE (log): {rmse:.4f}")
    print(f"Spearman r: {corr.correlation:.4f}")
    fold_changes = np.abs(true_log - imputed_log)
    print(f"Within 0.5 log: {(fold_changes < 0.5).mean()*100:.1f}%")
    print(f"Within 1.0 log: {(fold_changes < 1.0).mean()*100:.1f}%")
    print(f"Within 2.0 log: {(fold_changes < 2.0).mean()*100:.1f}%")
    return mae, rmse, corr.correlation

def evaluate_zero_classification(X_true, X_imputed, holdout_idx, threshold=1e-6):
    """
    Evaluate how well 0s are preserved during imputation
    threshold: values below this are treated as zero (for numerical stability)
    """
    true_vals = X_true[holdout_idx[:, 0], holdout_idx[:, 1]]
    imputed_vals = X_imputed[holdout_idx[:, 0], holdout_idx[:, 1]]

    # Define zeros
    true_zero = true_vals == 0
    imputed_zero = imputed_vals < threshold
    # Total true zeros
    total_true_zeros = true_zero.sum()
    
    if total_true_zeros == 0:
        print("No true zeros in held-out set.")
        return None
    
    # False positives: true zero -> imputed non-zero
    false_zero = true_zero & (~imputed_zero)
    false_zero_pct = false_zero.sum() / total_true_zeros * 100
    
    print("\nZero Classification")
    print(f"Total true zeros: {total_true_zeros}")
    print(f"False non-zero imputations: {false_zero.sum()}")
    print(f"Percentage falsely classified zeros (PFC): {false_zero_pct:.2f}%")
    return false_zero_pct

def mean_variance(X_true, X_imputed, mask):
    """
    Mean-Variance relationship

    NB: variance = mean + mean^2/dispersion
    Variance should increase faster than the mean (overdispersion)

    Gaussian: variance = constant(sigma^2)
    Variance is independent of mean

    Mean-variance relationship of imputed data should match the original mean-variance relationship.
    If imputation introduces Gaussian noise, the relationship will flatten and variance will not scale with mean^2.
    DESeq2 estimates dispersion from this mean-variance relationship. 
    So, if mean-variance relationship is presereved, biologigical fidelity is also preserved.

    Slope (log-log regression):
    ~1: Poisson (variance is roughly the same as mean)
    ~2: NB (variance is roughly the same as mean^2 (no dispersion))
    <1: underdispersed
    >1: overdispersed
    """
    print("\nMean-Variance Relationship")

    X_obs_only = np.where(mask == 1, X_true, np.nan) # replace missing values with NaN
    
    # Mean and Variance per gene from original data
    obs_mean = np.nanmean(X_obs_only, axis=0) # mean
    obs_var = np.nanvar(X_obs_only,  axis=0) # variance

    # Mean and Variance per gene from imputed data  
    imp_mean = X_imputed.mean(axis=0)
    imp_var = X_imputed.var(axis=0)  

    # Log regression requires positive values
    valid_obs = (obs_mean > 0) & (obs_var > 0)
    valid_imp = (imp_mean > 0) & (imp_var > 0)

    # Fit log-log regression
    # log(var) = slope * log(mean) + constant (y = mx + c)
    obs_slope = np.polyfit(np.log(obs_mean[valid_obs]), np.log(obs_var[valid_obs]), 1)[0]
    imp_slope = np.polyfit(np.log(imp_mean[valid_imp]), np.log(imp_var[valid_imp]), 1)[0]

    print(f"Mean-Variance Slope (Observed): {obs_slope:.3f}")
    print(f"Mean-Variance Slope (Imputed): {imp_slope:.3f}")

    # Correlation between mean and variance between observed and imputed data
    valid_both = valid_obs & valid_imp

    mean_corr = spearmanr(obs_mean[valid_both], imp_mean[valid_both], nan_policy='omit')
    var_corr  = spearmanr(obs_var[valid_both], imp_var[valid_both], nan_policy='omit')

    print(f"Per-Gene Mean Spearman Correlation (Obs vs Imp): {mean_corr.correlation:.4f}, P-value: {mean_corr.pvalue:.4f}")
    print(f"Per-Gene Variance Spearman Correlation (Obs vs Imp): {var_corr.correlation:.4f}, P-value: {var_corr.pvalue:.4f}")

    return obs_slope, imp_slope, mean_corr, var_corr

def dispersion_stability(X_true, X_imputed, mask):
    """
    Dispersion estimate stability

    dispersion = mean^2 / (variance - mean) 

    If variance is <= mean: underdispered or Poisson, dispersion is undefined
    Imputed values that are poorly calibrate can produce extreme dispersion estimates, 
    undefined dispersion (imputation reduced variance),
    or high variance across similar genes

    Fold-change: calculates how much dispersion estimates change after imputation using log-scale difference
        = |log(r_imp) - log(r_obs)|
    Median fold-change:
    < 0.5: stable, imputation preserves dispersion
    0.5-1.0: some genes affected
    > 1.0: unstable, imputation distorts dispersion
    """
    print("\nDispersion Estimate Stability")

    X_obs_only = np.where(mask == 1, X_true.astype(float), np.nan)

    def estimate_dispersion(X, min_mean=0.1):
        mu = np.nanmean(X, axis=0) # mean per gene
        var = np.nanvar(X, axis=0) # variance per gene
        valid = (var > mu) & (mu > min_mean) # valid dispersion (not undefined)
        r = np.full(X.shape[1], np.nan) # create dispersion array
        r[valid] = mu[valid] ** 2 / (var[valid] - mu[valid]) # calculate dispersion per gene
        return r, valid
    
    # Per-gene dispersion estimate of observed data
    r_obs, valid_obs = estimate_dispersion(X_obs_only)

    # Per-gene dispersion estimate of imputed data
    r_imp, valid_imp = estimate_dispersion(X_imputed.astype(float))

    print(f"Genes with defined dispersion (Observed): {valid_obs.sum()}")
    print(f"Genes with defined dispersion (Imputed): {valid_imp.sum()}")

    # Genes where imputation made dispersion undefined (doesn't included genes that became valid after imputation)
    new_undefined = valid_obs & ~valid_imp

    
    # For genes with defined dispersion in both (again does not include gene that gained defined dispersion)
    # Correlation between dispersion for these genes
    both_valid = valid_obs & valid_imp

    results = {
        'disp_n_obs': int(valid_obs.sum()),
        'disp_n_imp': int(valid_imp.sum()),
        'disp_n_lost': int(new_undefined.sum()),
        'disp_n_both': int(both_valid.sum()),
        'disp_corr': np.nan,
        'disp_median_lfc': np.nan,
        'disp_n_extreme': np.nan
    }

    if both_valid.sum() > 0:
        r_obs_both = r_obs[both_valid]
        r_imp_both = r_imp[both_valid]

        r_corr = spearmanr(r_obs_both, r_imp_both)
        # Median fold-change
        fold_change = np.median(np.abs(np.log2(r_imp_both + 1e-8) - np.log(r_obs_both + 1e-8)))

        print(f"Genes losing defined dispersion after imputation: {new_undefined.sum()} ({(new_undefined.sum()/valid_obs.sum())*100:.1f}%)") # imputation reduced variance
        print(f"Genes maintaining defined dispersion: {both_valid.sum()}")
        print(f"Dispersion Correlation (Obs vs Imp): {r_corr.correlation:.4f}, P-value: {r_corr.pvalue:.4f}")
        print(f"Median log fold change in dispersion: {fold_change:.4f}")

        # Flag extreme dispersion genes, FC > 2
        extreme = np.abs(np.log(r_imp_both + 1e-8) - np.log(r_obs_both + 1e-8)) > 2.0
        print(f"Genes with >2 log fold change in dispersion: {extreme.sum()}")

        results.update({
            'disp_corr':       r_corr.correlation,
            'disp_median_lfc': fold_change,
            'disp_n_extreme':  int(extreme.sum())
        })
        
    return results

def stratified_evaluation(X_true, X_imputed, holdout_idx):
    """
    Stratified evaluation by expression level
    returns log mae and spearman by Q1 (low expression), Q2, Q3, Q4 (high expression)
    """
    true_vals = X_true[holdout_idx[:, 0], holdout_idx[:, 1]].astype(float)
    imputed_vals = X_imputed[holdout_idx[:, 0], holdout_idx[:, 1]].astype(float)

    # Convert to log space
    true_log = np.log1p(true_vals)
    imp_log  = np.log1p(imputed_vals)

    breakpoints = np.quantile(true_log, [0.0, 0.25, 0.50, 0.75, 1.0])
    bin_labels  = ['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)']
    bins = np.digitize(true_log, breakpoints[1:-1])

    results = {}
    for i, label in enumerate(bin_labels):
        mask_bin = bins == i
        if mask_bin.sum() < 5:
            for metric in ['MAE', 'Spearman']:
                results[f'{label}_{metric}'] = np.nan
            continue

        t = true_log[mask_bin]
        p = imp_log[mask_bin]

        mae = np.mean(np.abs(t - p))
        spearman = spearmanr(t, p).correlation


        results[f'{label}_MAE'] = round(mae, 4)
        results[f'{label}_Spearman'] = round(spearman, 4)
        
    return results

In [ ]:
# Data loading and preprocessing
df = pd.read_csv("countdata5.txt", sep=r'\s+', index_col=0) # change per dataset
df = df.T # transpose (rows = samples, columns = genes)
df = df.replace('NA', np.nan) # replace NA with numpy NaN
df = df.astype(float) # data as float

print(f"Data shape: {df.shape} (samples x genes)")

# Filter genes
keep_genes = (
    (df.notna().sum(axis=0) > 10) & # >10 observed values
    ((df > 0).sum(axis=0) > 5) # >5 nonzero values
)

df = df.loc[:, keep_genes]

# Raw counts, used for NB likelihood and validation
mask  = (~df.isnull()).astype(float).values # observed = 1, missing = 0
X_raw = df.fillna(0).values.astype(np.float32) # fill missing with 0

# Library size normalised + log1p, used as encoder input only
"""
Median of each genes is used instead of mean to prevent outlier samples distorting scale
X values normalised using median, (+1 avoids division by 0)
Transform X values to log1p
X_input only used in encoder for stability
"""
library_sizes  = X_raw.sum(axis=1, keepdims=True)
median_library = np.median(library_sizes)
X_norm         = (X_raw / (library_sizes + 1)) * median_library
X_input        = np.log1p(X_norm).astype(np.float32)
X_input        = np.where(mask == 1, X_input, 0.0).astype(np.float32)

# Validation mask
train_mask, holdout_idx = create_validation_mask(mask, holdout_fraction=0.1)
print(f"Number of genes after filtering: {df.shape[1]}")
print(f"Holdout positions: {len(holdout_idx)}")
print(f"X_raw range: {X_raw.min():.0f} to {X_raw.max():.0f}")
print(f"X_input range (norm and log1p): {X_input.min():.2f} to {X_input.max():.2f}")

In [ ]:
# Validate
def evaluate_all(X_true, X_imputed, mask, holdout_idx):
    """
    Full evaluation pipeline returning a results dictionary
    """
    mae, rmse, nrmse, spearman = evaluate_imputation(X_true, X_imputed, holdout_idx)
    mae_nz, rmse_nz, nrmse_nz, spearman_nz = evaluate_imputation_nonzero(X_true, X_imputed, holdout_idx)
    pfc = evaluate_zero_classification(X_true, X_imputed, holdout_idx)
    log_mae, log_rmse, log_spearman = evaluate_imputation_logspace(X_true, X_imputed, holdout_idx)
    obs_slope, imp_slope, mean_corr, var_corr = mean_variance(X_true, X_imputed, mask)
    disp_metrics = dispersion_stability(X_true, X_imputed, mask)
    strat = stratified_evaluation(X_true, X_imputed, holdout_idx)
    return {
        # Overall metrics
        'mae': mae,
        'rmse': rmse,
        'nrmse': nrmse,
        'spearman': spearman,
        # Non-zero metrics
        'mae_nz':mae_nz,
        'rmse_nz':rmse_nz,
        'spearman_nz':spearman_nz,
        # Zero classification
        'pfc':pfc,
        # Log space
        'log_mae':log_mae,
        'log_rmse':log_rmse,
        'log_spearman':log_spearman,
        # Mean-variance
        'obs_slope':obs_slope,
        'imp_slope':imp_slope,
        'slope_delta':abs(obs_slope - imp_slope),
        # Dispersion 
        'disp_n_lost':disp_metrics['disp_n_lost'],
        'disp_median_lfc': disp_metrics['disp_median_lfc'],
        'disp_n_extreme': disp_metrics['disp_n_extreme'],
        'disp_corr':disp_metrics['disp_corr'],
        # Stratified evaluation
        'Q1_MAE': strat['Q1 (low)_MAE'],
        'Q2_MAE': strat['Q2_MAE'],
        'Q3_MAE': strat['Q3_MAE'],
        'Q4_MAE': strat['Q4 (high)_MAE'],
        # Stratified Spearman
        'Q1_Spearman': strat['Q1 (low)_Spearman'],
        'Q2_Spearman': strat['Q2_Spearman'],
        'Q3_Spearman': strat['Q3_Spearman'],
        'Q4_Spearman': strat['Q4 (high)_Spearman']
    }

In [ ]:
# Grid search to refine hyper parameters (Runs for atleast 2 hours)
param_grid = {
    'latent_size': [16, 32],
    'hidden_size': [128, 256, 512],
    'K': [100],
    'lr': [1e-4, 5e-4]
}

results = []

for latent, hidden, K, lr in itertools.product(*param_grid.values()):
    print(f"Training: latent={latent}, hidden={hidden}, K={K}, lr={lr}")

    # Initialise model
    model = MIWAE_NB(
        input_size = 39,
        latent_size = latent,
        hidden_size = hidden,
        K = K
    )

    # Train
    model = train_miwae_nb(
    X_input, X_raw, train_mask,
    epochs = 3000,
    lr = lr,
    batch_size = 16,
    latent_size = latent,
    hidden_size = hidden,
    K = K
)
    # Impute
    X_imputed = model.impute(
    torch.tensor(X_input, dtype=torch.float32).to(device),
    torch.tensor(X_raw, dtype=torch.float32).to(device),
    torch.tensor(train_mask, dtype=torch.float32).to(device),
    )
    X_imputed = np.round(np.clip(X_imputed, 0, None)).astype(int)
    
    # Results
    metrics = evaluate_all(X_raw, X_imputed, mask, holdout_idx)  # your existing evaluation function
    
    results.append({
        'latent_size': latent,
        'hidden_size': hidden,
        'K': K,
        'lr': lr,
        'disp_lost': metrics['disp_n_lost'],
        'disp_lfc': metrics['disp_median_lfc'],
        'pfc': metrics['pfc'],
        'log_mae': metrics['log_mae']
    })

    print(f"  disp_lost={metrics['disp_n_lost']}, pfc={metrics['pfc']:.2f}, log_mae={metrics['log_mae']:.4f}\n")

results_df = pd.DataFrame(results).sort_values('disp_lost')
print(results_df)

In [ ]:
# Train model
model = train_miwae_nb(
    X_input, X_raw, train_mask,
    epochs = 3000,
    lr = 0.0005,
    batch_size = 16,
    latent_size = 32,
    hidden_size = 512
)

In [ ]:
# Impute
X_imputed = model.impute(
    torch.tensor(X_input, dtype=torch.float32).to(device),
    torch.tensor(X_raw, dtype=torch.float32).to(device),
    torch.tensor(train_mask, dtype=torch.float32).to(device),
)
X_imputed = np.round(np.clip(X_imputed, 0, None)).astype(int)

In [ ]:
# Evaluation results and produce results csv
import os

output_path = 'imputation_metrics_miwae_log_FINAL.csv'
dataset_label = '30% Dropout' # change per dataset

metrics = evaluate_all(X_raw, X_imputed, mask, holdout_idx)

row_df = pd.DataFrame([metrics])
row_df.insert(0, 'Dataset', dataset_label)

# Append to CSV (creates file on first run, appends on subsequent runs)
if os.path.exists(output_path):
    row_df.to_csv(output_path, mode='a', header=False, index=False)
    print(f"Appended {dataset_label} to {output_path}")
else:
    row_df.to_csv(output_path, mode='w', header=True, index=False)
    print(f"Created {output_path} with {dataset_label}")

In [ ]:
# Plot true values vs imputed values of held out test
true_vals = X_raw[holdout_idx[:, 0], holdout_idx[:, 1]].astype(float)
imputed_vals = X_imputed[holdout_idx[:, 0], holdout_idx[:, 1]].astype(float)

nonzero = true_vals > 0
true_nz = np.log1p(true_vals[nonzero])
imputed_nz = np.log1p(imputed_vals[nonzero])

plt.figure(figsize=(6, 6))
plt.scatter(true_nz, imputed_nz, alpha=0.3, s=2, color='steelblue')
lim = max(true_nz.max(), imputed_nz.max())
plt.plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfect imputation')
plt.xlabel('True log1p(counts)')
plt.ylabel('Imputed log1p(counts)')
plt.title('True vs Imputed Values (Log Space, Non-Zero)')
plt.legend()
plt.tight_layout()
plt.savefig('true_vs_imputed_log.png', dpi=100)
plt.show()

In [ ]:
# Run Imputation and save as csv

model.eval()

# Run imputation without held mask
imputed_raw = model.impute(
    torch.tensor(X_input, dtype=torch.float32).to(device),
    torch.tensor(X_raw, dtype=torch.float32).to(device),
    torch.tensor(train_mask, dtype=torch.float32).to(device),
)

# Round to integers and clip negatives for DESeq2
final_counts = np.clip(np.round(imputed_raw), 0, None).astype(int)

gene_names = df.columns.tolist()
sample_names = df.index.tolist()

final_df = pd.DataFrame(
    final_counts,
    index=sample_names,
    columns=gene_names
).T 

final_df.to_csv("tuned_DESeq2_Ready_miwae_log_5pct.csv") # change per dataset